# Случайный лес

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import joblib
import pickle

```
Преобразование категориальных переменных
```

In [3]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг',
               'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [4]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

In [5]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        verbose=2
    ))
])

```
Загрузка обучающей и тестовой выборок
```

In [6]:
X_train, X_test, y_train, y_test = joblib.load("data/data_split.pkl")

```
Обучение и сохранение модели
```

In [7]:
model.fit(X_train, y_train)
joblib.dump(model, "models/rf_model.pkl")

['models/rf_model.pkl']

```
Предсказание на тестовой выборке
```

In [8]:
y_pred = model.predict(X_test)

[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    0.2s
[Parallel(n_jobs=6)]: Done  50 out of  50 | elapsed:    0.4s finished


```
Оценка модели
```

In [9]:
rf_mse = mean_squared_error(y_test, y_pred)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test, y_pred)
rf_r2 = r2_score(y_test, y_pred)

```
Вывод результатов
```

In [10]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [rf_mse, rf_rmse, rf_mae, rf_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),96_323_832_172.77
1,Среднеквадратическая ошибка (RMSE),310_360.81
2,Средняя абсолютная ошибка (MAE),190_047.58
3,Коэффицент детерминации (R^2),0.91


```
Сохранение метрик в отдельный файл
```

In [11]:
metrics = {
    "model_name": "Random Forest",
    "MSE": rf_mse,
    "RMSE": rf_rmse,
    "MAE": rf_mae,
    "R2": rf_r2
}

In [12]:
with open("metrics/rf_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)